# Reproducing the BehaviorGPT PixelRec benchmark through the API

Our paper reports how well **BehaviorGPT**, zero-shot, predicts the next item on
[PixelRec](https://github.com/westlake-repl/PixelRec). This notebook reproduces that evaluation
end to end with public tools: it builds the benchmark split from the official PixelRec download,
asks the hosted model for a ranking at every evaluated position through the public API, and computes
the metrics from the returned rankings in a few lines of numpy.

1. **Data**: download, preprocess and check the benchmark split.
2. **The task**: which positions are scored, what history each one sees, and what the candidates are.
3. **Predictions**: one API call by hand, then the full loop, saving every ranking to disk.
4. **Metrics**: recall, NDCG and MRR from the saved rankings.
5. **Results**: the numbers next to the paper's.

The reference numbers, on the 79,399 evaluated positions:

| | recall@1 | recall@10 | recall@100 | NDCG@10 | MRR@10 |
|---|---|---|---|---|---|
| BehaviorGPT, zero-shot, as in the paper | 0.00091 | 0.00594 | 0.03346 | 0.00293 | 0.00203 |
| the same, with one date for every position (2022-01-29) | 0.00062 | 0.00432 | 0.02684 | 0.00213 | 0.00147 |

The paper measures item ages and sales at each position's own date. A hosted catalog measures them once, at
the date it was embedded, so the closest internal comparison is our run with that same single date; the
notebook shows both.

Recall this low is expected for ranking a catalog of about 90,000 items for users the model has never
seen. A popularity ranking, computed below, is the floor.

## Before you start

- **An API key**: `UNBOXAI_API_KEY` in `.env`, as for the other notebooks.
- **The PixelRec data**, downloaded by hand from the official source. The
  [PixelRec repository](https://github.com/westlake-repl/PixelRec) links two Google Drive folders [Interactions](https://drive.google.com/drive/folders/1vR1lgQUZCy1cuhzPkM2q7AsdYRP43feQ?usp=drive_link) and [Item Information](https://drive.google.com/drive/folders/1rXBM-zi5sSdLHNshXWtGgVReWLuYNgDB?usp=drive_link) under
  *Download Link*. From each, download the file `Pixel200K.csv` (about 131 MB together) and place them as:

```
notebooks/data/pixelrec/raw/interactions/Pixel200K.csv   from the "Interaction" folder
notebooks/data/pixelrec/raw/items/Pixel200K.csv          from the "Item Information" folder
```

  Both files have the same name, so keep them in these two folders; the setup cell below creates them.
- **Time**: the full evaluation is one API call per position, about 80,000 calls, roughly an hour with
  the default 8 parallel workers. It saves as it goes and resumes where it stopped. By default `MAX_CALLS`
  scores a fixed random subset of 5,000 positions, a few minutes; set it to `None` for the full run.

## What this notebook writes to your disk

| what | where | size |
|---|---|---|
| the two downloaded CSVs (you place them) | `notebooks/data/pixelrec/raw/` | 131 MB |
| the preprocessed benchmark split: sequences, catalog and the API catalog | `notebooks/data/pixelrec/` | about 115 MB |
| the saved rankings, one row per scored position | `notebooks/results/pixelrec/` | about 95 MB for the full run |

Both folders are ignored by git. To put the data somewhere else, set `BEHAVIORGPT_DATA_DIR` in `.env` (a
relative path is relative to `notebooks/`); the rankings go to `RESULTS_DIR` in the next cell.

## Setup

In [ ]:
import hashlib
import json
import os
import random
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from datetime import UTC, datetime, timedelta
from pathlib import Path

import httpx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
from dotenv import load_dotenv
from IPython.display import display
from tqdm import tqdm

from behaviorgpt import RateLimitError, UnboxAIClient, UnboxAIError, View
from behaviorgpt.resources.catalogs import CATALOG_SCHEMA, check_catalog_schema
from behaviorgpt.types import Domains, SessionEvent

In [ ]:
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "prepare_pixelrec.py").exists())
load_dotenv(ROOT / ".env", override=True)

DATA_DIR = ROOT / "notebooks" / (os.environ.get("BEHAVIORGPT_DATA_DIR") or "data") / "pixelrec"
RAW_DIR = DATA_DIR / "raw"  # the two downloads go here
RESULTS_DIR = ROOT / "notebooks" / "results" / "pixelrec"  # the saved rankings
for folder in (RAW_DIR / "interactions", RAW_DIR / "items", RESULTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

CATALOG_ID = "cat_77d554465cbded0dc963769f"  # PixelRec200K, hosted for every API key

N_POSITIONS = 79_399  # the positions the evaluation scored
PAPER = {  # the paper's numbers: each position's item ages and sales measured at its own date
    "recall@1": 0.00090681, "recall@5": 0.00322422, "recall@10": 0.00594466,
    "recall@20": 0.00992456, "recall@50": 0.02013879, "recall@100": 0.03346390,
    "ndcg@10": 0.00292520, "mrr@10": 0.00202859,
}
SAME_DATE = {  # the internal run with one reference date for every position, 2022-01-29
    "recall@1": 0.00061714, "recall@5": 0.00254411, "recall@10": 0.00431995,
    "recall@20": 0.00772050, "recall@50": 0.01586922, "recall@100": 0.02683913,
    "ndcg@10": 0.00212662, "mrr@10": 0.00147039,
}

TIMEZONE = "UTC"
BLOCK = 512  # events the model reads at once
MAX_K = 100  # ranking depth requested per call

client = UnboxAIClient(market="unk", timezone=TIMEZONE)

### Settings you can change

In [ ]:
NUM_CANDIDATES = 5_000  # nearest-neighbor exploration size; the server's maximum
KS = (1, 5, 10, 20, 50, 100)  # depths the metrics are reported at; the paper's maximum is 100
PLOT_KS = (1, 10, 100)  # the depths drawn in the plot; the tables show every depth
MAX_WORKERS = 8
MAX_CALLS: int | None = 5_000  # None scores every position; a number scores a fixed random subset

assert MAX_K <= NUM_CANDIDATES <= 5_000, "NUM_CANDIDATES must be between MAX_K and the server's maximum, 5,000"
assert max(KS) <= MAX_K, "every depth in KS must be at most MAX_K"

## 1. Data

### From the downloads to the benchmark split

`scripts/prepare_pixelrec.py` builds the split our evaluation used. It is plain pyarrow and worth reading
alongside this notebook:

- a session is one user's views on one UTC day;
- users are hashed into train (90 %) and test (10 %);
- the cutoff is the day by which 80 % of all views have happened (2021-11-12): train keeps the sessions on
  or before it, test keeps the sessions after it, so every evaluated view comes after everything the
  training period contains;
- the catalog records, per item, the day it was first seen and how it sold in the training period.

It runs once, in a minute or two, and is skipped when its output already exists. Before it runs, the two
downloads are checked against the SHA-256 fingerprints of the release our numbers were measured on.

In [ ]:
RAW_SHA256 = {  # the release our numbers were measured on
    RAW_DIR / "interactions" / "Pixel200K.csv": "77daac2c79079a7b389ca8c0f697b63a92ec981f1c61a5e98e94790fee399ac7",
    RAW_DIR / "items" / "Pixel200K.csv": "00d039fccaafc4aa1356379fc8688b6fa7d883b4331893c7827064986d435244",
}
if not all((DATA_DIR / name).exists() for name in ("meta.json", "sequences/test", "catalog", "catalog_api.parquet")):
    missing = [str(path) for path in RAW_SHA256 if not path.exists()]
    if missing:
        raise FileNotFoundError(f"download the PixelRec files (see Before you start) and place them as: {missing}")
    for path, expected in RAW_SHA256.items():
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError(f"{path} differs from the release our numbers were measured on")
    script = ROOT / "scripts" / "prepare_pixelrec.py"
    subprocess.run([sys.executable, str(script), "--raw", str(RAW_DIR), "--out", str(DATA_DIR)], check=True)

meta = json.loads((DATA_DIR / "meta.json").read_text())
counts = meta["counts"]
print(f"Cutoff day: {meta['cutoff']}.")
print(f"Users: {counts['train_users']:,} train and {counts['test_users']:,} test; "
      f"{sum(meta['dropped_users'].values()):,} had no session on their side and were dropped.")
print(f"Items: {counts['items']:,}, of which {counts['items_in_train']:,} were viewed in the training period.")

### The sequences

One row per user. The row is a single string: the user id, then the user's sessions in time order. A
session is nine bracketed *domain* tokens (who and when), the items the user viewed that day, and
`<end>`. Anything that is not `[bracketed]`, `"quoted"` or `<special>` is an item id.

In [ ]:
shards = sorted((DATA_DIR / "sequences" / "test").glob("*.parquet"))
users = pd.concat(pd.read_parquet(p, columns=["user_id", "text"]) for p in shards)
print(f"{len(users):,} test users. The first row:\n")
print(users.text.iloc[0])

In [ ]:
MONTHS = {m: i for i, m in enumerate("Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec".split(), 1)}
DOMAIN_NAMES = ("gender", "age_group", "postal_code", "market", "year", "month", "day", "weekday", "hour")
PROFILE = DOMAIN_NAMES[:4]  # the who; the rest is the when


@dataclass(frozen=True)
class Session:
    start: datetime
    items: tuple[str, ...]
    profile: dict[str, str]  # gender, age_group, postal_code, market as written in the data


def is_item(token: str) -> bool:
    """A token that is not [bracketed], "quoted" or <special> is an item id."""
    return token[0] not in '["<' and token[-1] not in ']">'


def domain_values(tokens: list[str]) -> dict[str, str]:
    """Name the nine domain tokens; `[year:2021]` gives 2021 and a bare `[Nov]` gives Nov."""
    return {name: tok[1:-1].split(":")[-1] for name, tok in zip(DOMAIN_NAMES, tokens)}


def session_start(domains: dict[str, str]) -> datetime:
    """The session's day and hour as a UTC datetime."""
    hour = 0 if domains["hour"] == "unk" else int(domains["hour"])
    return datetime(int(domains["year"]), MONTHS[domains["month"]], int(domains["day"]), hour, tzinfo=UTC)


def parse_user(text: str) -> tuple[str, list[Session]]:
    """Split one row into its user id and its sessions."""
    user_id, body = text.split(", ", 1)
    sessions, tokens = [], []
    for token in body.split():
        if token != "<end>":
            tokens.append(token)
            continue
        domains, rest = domain_values(tokens[: len(DOMAIN_NAMES)]), tokens[len(DOMAIN_NAMES) :]
        items = tuple(t for t in rest if is_item(t))
        sessions.append(Session(session_start(domains), items, {name: domains[name] for name in PROFILE}))
        tokens = []
    return user_id, sessions


uid, sessions = parse_user(users.text.iloc[0])
print(f"User {uid} has {len(sessions)} session{'' if len(sessions) == 1 else 's'}:")
for s in sessions[:5]:
    print(f"  {s.start:%Y-%m-%d %H:%M}  {' '.join(s.items)}")

### Checks against our build

A few rows copied from the split our numbers were measured on: one user with a single view, one with three
views on the same day, and one with views on three days. If any of them differs, the split differs.

In [ ]:
GOLDEN = {
    "u10003984": "u10003984, [gender:unk] [age_group:unk] [postal_code:unk] [market:unk] [year:2022] [Jan] [day:22] [Sat] [hour:15] i130041 <end>",
    "u3373012": "u3373012, [gender:unk] [age_group:unk] [postal_code:unk] [market:unk] [year:2022] [Mar] [day:28] [Mon] [hour:11] i129244 i237091 i111498 <end>",
    "u10177958": "u10177958, [gender:unk] [age_group:unk] [postal_code:unk] [market:unk] [year:2022] [Jan] [day:25] [Tue] [hour:16] i195980 <end> "
                 "[gender:unk] [age_group:unk] [postal_code:unk] [market:unk] [year:2022] [Feb] [day:7] [Mon] [hour:4] i30307 i145666 <end> "
                 "[gender:unk] [age_group:unk] [postal_code:unk] [market:unk] [year:2022] [Feb] [day:26] [Sat] [hour:16] i222503 <end>",
}
texts = dict(zip(users.user_id, users.text))
for user_id, expected in GOLDEN.items():
    assert texts[user_id] == expected, f"{user_id} differs from our build"
SESSIONS = dict(parse_user(t) for t in users.text)
assert SESSIONS["u10177958"][1] == Session(datetime(2022, 2, 7, 4, tzinfo=UTC), ("i30307", "i145666"),
                                           dict.fromkeys(PROFILE, "unk"))
print("The sequences match our build.")

## 2. The task

### What counts as a position

The model reads a user as one stream of events: a session marker, then that session's items, then the next
marker, and so on. **Every event whose successor is an item is a position**: the events up to and including
it are the history, and the successor is the answer we hold out. A user with ten views yields about ten
positions, not one.

Two details decide how much history a position sees:

- The stream is cut into **blocks of 512 events**. A position sees only the events earlier in its own
  block, so the first position of every block, and every position at the very start of a user, has an
  empty history. Those are scored too.
- The **last slot of a block has no successor** inside the block and is skipped.

```
stream:   <sep> A B <sep> C <sep> D E ...
position:       ^ history=[]     answer=A
                  ^ history=[A]  answer=B
                      ^ history=[A,B]  (next is <sep>, not scored)
                        ^ history=[A,B]  answer=C
```

In [ ]:
@dataclass(frozen=True)
class Position:
    pos_id: int
    user_id: str
    token_index: int  # index of the context event in the user's stream; with user_id it names the position
    history: tuple[tuple[int, str], ...]  # (session index, item id), oldest first
    gold: str
    gold_session: int  # session index of the answer; its start is the time we predict at
    at_session_start: bool


def token_stream(sessions: list[Session]) -> list[tuple[int, str]]:
    """One marker per session followed by its items, as the model reads them."""
    return [(i, tok) for i, s in enumerate(sessions) for tok in ("<sep>", *s.items)]


def iter_positions(user_id: str, sessions: list[Session], start_id: int) -> list[Position]:
    """Every position whose next event is an item, with the history visible from it."""
    stream, out = token_stream(sessions), []
    for j in range(len(stream) - 1):
        if j % BLOCK == BLOCK - 1 or stream[j + 1][1] == "<sep>":
            continue
        block = stream[BLOCK * (j // BLOCK) : j + 1]
        history = tuple(t for t in block if t[1] != "<sep>")
        gold_session, gold = stream[j + 1]
        out.append(Position(start_id + len(out), user_id, j, history, gold, gold_session, stream[j][1] == "<sep>"))
    return out


positions = []
for user_id, sessions in SESSIONS.items():
    positions += iter_positions(user_id, sessions, len(positions))

The evaluation scored 79,399 positions. This build must produce the same set, and the user with views on
three days pins down the rule: four positions, each with the history visible from it.

In [ ]:
assert len(positions) == N_POSITIONS, f"{len(positions):,} positions, expected {N_POSITIONS:,}"
got = [(p.gold, [item for _, item in p.history]) for p in positions if p.user_id == "u10177958"]
assert got == [("i195980", []), ("i30307", ["i195980"]),
               ("i145666", ["i195980", "i30307"]), ("i222503", ["i195980", "i30307", "i145666"])]

n_hist = pd.Series([len(p.history) for p in positions])
print(f"{len(positions):,} positions: {(n_hist == 0).mean():.1%} have no history and "
      f"{np.mean([p.at_session_start for p in positions]):.1%} are at the start of a session.")
n_hist.describe(percentiles=[0.5, 0.9, 0.99])[["mean", "50%", "90%", "99%", "max"]].round(1).rename(
    {"50%": "median", "90%": "p90", "99%": "p99"}).to_frame("history length").T

### The candidates

The model ranks every item it has embedded, so the embedded catalog is the candidate set: **every item that
occurs in the sequences**, train or test, 89,437 in all. The catalog is hosted on the server as
`PixelRec200K`, embedded with 2022-01-29 as the date item ages and sales are measured at. The local catalog
built above is read to list the candidates, for the popularity floor, and to check that the hosted catalog
holds our items.

In [ ]:
def read_catalog(catalog_dir: Path) -> pa.Table:
    """Concatenate the Arrow shards of a catalog, keeping the columns the API reads."""
    tables = [pa.ipc.open_file(p).read_all() for p in sorted(catalog_dir.glob("*.arrow"))]
    return pa.concat_tables(tables).select(list(CATALOG_SCHEMA))


def item_ids(split: str) -> set[str]:
    """Every item id that occurs in a split's sequences."""
    ids: set[str] = set()
    for p in sorted((DATA_DIR / "sequences" / split).glob("*.parquet")):
        for text in pd.read_parquet(p, columns=["text"]).text:
            ids.update(item for s in parse_user(text)[1] for item in s.items)
    return ids


candidates = item_ids("train") | item_ids("test")
catalog = read_catalog(DATA_DIR / "catalog")
catalog = catalog.filter(pc.is_in(catalog["id"], value_set=pa.array(sorted(candidates))))
CATALOG_IDS = set(catalog.column("id").to_pylist())
assert len(CATALOG_IDS) == 89_437, f"{len(CATALOG_IDS):,} candidates, expected 89,437"
assert all(p.gold in CATALOG_IDS for p in positions)

row = catalog.filter(pc.equal(catalog.column("id"), "i192714")).to_pylist()[0]  # one row checked against our build
assert (row["categories"], row["timestamp"], row["frequency"]) == ("Pet Reptiles", 1586131200, 70)
assert row["sales_since"] == [17, 24, 30, 36, 37, 40, 42, 42, 42, 44, 46, 70]

print(f"{len(CATALOG_IDS):,} candidates. The first three:")
preview = catalog.slice(0, 3).to_pandas()[["id", "categories", "timestamp", "frequency"]]
preview.assign(timestamp=pd.to_datetime(preview.timestamp, unit="s").dt.date).rename(
    columns={"timestamp": "first seen", "frequency": "training views"})

The script also writes `catalog_api.parquet`: the same candidates in the format the API's embed endpoint
takes, with only the columns the endpoint reads. `PixelRec200K` was embedded from this file, with cover
image URLs added. Nothing is uploaded from this notebook; embedding a catalog of your own is covered in
`showcase.ipynb`.

In [ ]:
api_catalog = pd.read_parquet(DATA_DIR / "catalog_api.parquet")
check_catalog_schema(DATA_DIR / "catalog_api.parquet")  # raises if a column is missing or mistyped
assert set(api_catalog.id) == CATALOG_IDS
print(f"The API catalog holds {len(api_catalog):,} items in the {len(api_catalog.columns)} columns the API reads.")

In [ ]:
def in_catalog(item_id: str) -> bool:
    """Whether the hosted catalog holds the item; any error other than "not found" is raised."""
    try:
        return bool(client.similar_items(item_id, limit=1, catalog_id=CATALOG_ID).items)
    except UnboxAIError as exc:
        if exc.status_code == 404:
            return False
        raise


sample = random.Random(0).sample(sorted(CATALOG_IDS), 100)
found = sum(map(in_catalog, sample))
assert found == len(sample), f"only {found} of {len(sample)} sampled candidates are in the hosted catalog"
print(f"All {len(sample)} sampled candidates are in the hosted catalog. Its top 5 for a user with no history:")
cold_start = client.complete(history=[], catalog_id=CATALOG_ID, limit=5).to_pandas()
cold_start[["id", "categories", "score"]].round(3)

## 3. Predictions

### One call by hand

`client.complete` stamps every event with the current time, which suits a live shopper; these sessions
happened in 2021 and 2022, so we build the request ourselves:

- Every item in the history becomes one timestamped `View` event. Each session keeps its own
  `session_id` and the user profile written in the data (`unk` throughout PixelRec).
- The request carries its own `timestamp` and `domains`: the moment the prediction is made, which is the
  start of the answer's session or just after the last event before the answer, and that session's profile.
- We ask for the top 100 and set `num_candidates` to 5,000 for the nearest neighbor search.

Two hand-picked positions show the result, both at the start of a session with three items of history: one
where the model's top 100 contains the answer, and one where it does not.

In [ ]:
def to_session_events(position: Position) -> list[SessionEvent]:
    """Wrap the visible history as timestamped view events, one session id per day."""
    sessions, per_session = SESSIONS[position.user_id], {}
    for i, item in position.history:
        per_session.setdefault(i, []).append(item)
    events = []
    for i, items in per_session.items():
        s = sessions[i]
        events += [  # a millisecond apart keeps the within-day order under any sort
            SessionEvent(store_id=CATALOG_ID, domains=Domains(**s.profile), timezone=TIMEZONE,
                         session_id=f"{position.user_id}:{i}", timestamp=s.start + timedelta(milliseconds=k),
                         event=View(item))
            for k, item in enumerate(items)
        ]
    return events


def prediction_context(position: Position, history: list[SessionEvent]) -> tuple[datetime, Domains]:
    """When and for whom the prediction is made: the answer session's start and profile."""
    s = SESSIONS[position.user_id][position.gold_session]
    # the server drops history events at or after the prediction time, so it must come after all of them
    now = max(s.start, history[-1].timestamp + timedelta(milliseconds=1)) if history else s.start
    return now, Domains(**s.profile)


def rank_items(history: list[SessionEvent], now: datetime, domains: Domains) -> tuple[list[str], list[float]]:
    """Ask the model what comes next at time `now`: the ranked item ids and their scores."""
    request = client.requester.make(history=history, catalog_id=CATALOG_ID, limit=MAX_K, domains=domains,
                                    timestamp=now, num_candidates=NUM_CANDIDATES)
    response = client.recommendations.create(payload=request, headers={"x-catalog-id": CATALOG_ID})
    return [item.id for item in response.items], [item.score for item in response.items]


EXAMPLES = {"found": ("u118189", 6), "not found": ("u10057654", 6)}  # (user, token index)
by_key = {(p.user_id, p.token_index): p for p in positions}
categories = dict(zip(catalog.column("id").to_pylist(), catalog.column("categories").to_pylist()))


def example_table(position: Position, events: list[SessionEvent], ranked: list[str]) -> pd.DataFrame:
    """The history, the answer and the top 3 returned, one row each."""
    rows = [("history", e.event.item, f"{e.timestamp:%Y-%m-%d %H:%M}") for e in events]
    rows.append(("answer", position.gold, ""))
    rows += [(f"ranked {k}", item, "") for k, item in enumerate(ranked[:3], start=1)]
    table = pd.DataFrame(rows, columns=["", "item", "viewed"])
    table["category"] = table["item"].map(categories)
    table["is the answer"] = np.where((table["item"] == position.gold) & table[""].str.startswith("ranked"), "yes", "")
    return table.set_index("").rename_axis(None)


for label, key in EXAMPLES.items():
    position = by_key[key]
    events = to_session_events(position)
    now, domains = prediction_context(position, events)
    ranked, _ = rank_items(events, now, domains)
    outcome = (f"ranked {ranked.index(position.gold) + 1} of {MAX_K}" if position.gold in ranked
               else f"not in the top {MAX_K}")
    print(f"{label.capitalize()}: user {position.user_id}, predicting on {now:%Y-%m-%d at %H:%M}. "
          f"The answer is {outcome}.")
    print(f"Request: {len(events)} view events, timestamp {now:%Y-%m-%d %H:%M:%S} UTC, limit {MAX_K}, "
          f"num_candidates {NUM_CANDIDATES:,}.")
    display(example_table(position, events, ranked))

### Scoring every position

One call per position, from a bounded thread pool. Rate limits (429), server errors and timeouts are
retried with exponential backoff and jitter; any other error is recorded on the row. A position whose call
failed is tried again on the next run and counts as a miss until then.

Each result is a small **frame** row: which position (`user_id`, `token_index`), the answer, and the full
top 100 with scores. Rows are appended to parquet files in chunks, so an interrupted run resumes where it
stopped, and nothing after this section needs another API call. Saved rows are checked against the
positions built above, so results from a different build are never mixed in.

With `MAX_CALLS` set, the first `MAX_CALLS` positions of one fixed random order are scored, so a larger
value extends the subset rather than replacing it, and every result below covers exactly that subset.

In [ ]:
RETRY_STATUS = {408, 429, 500, 502, 503, 504}


def with_retry(fn, attempts: int = 8, base: float = 1.0, cap: float = 60.0):
    """Call fn, backing off with jitter on rate limits, server errors and timeouts."""
    for attempt in range(attempts):
        try:
            return fn()
        except (RateLimitError, httpx.TransportError) as exc:
            error = exc
        except UnboxAIError as exc:
            if exc.status_code not in RETRY_STATUS:
                raise
            error = exc
        time.sleep(min(cap, base * 2**attempt) * random.uniform(0.5, 1.5))
    raise error


def score_position(position: Position) -> dict:
    """One API call, kept as a frame row: the position, its answer, and the full ranking returned."""
    started = time.perf_counter()
    try:
        events = to_session_events(position)
        ranked, scores = with_retry(lambda: rank_items(events, *prediction_context(position, events)))
        hit, error = first_hit(ranked, position.gold), None
    except Exception as exc:
        ranked, scores, hit, error = [], [], MAX_K, repr(exc)
    return {
        "pos_id": position.pos_id, "user_id": position.user_id, "token_index": position.token_index,
        "n_hist": len(position.history), "at_session_start": position.at_session_start, "gold": position.gold,
        "first_hit": hit, "top_k": ranked, "top_k_score": scores,
        "error": error, "latency_s": time.perf_counter() - started,
    }


def first_hit(ranked: list[str], gold: str) -> int:
    """0-based rank of the held-out item, or MAX_K when it is not in the list."""
    return ranked.index(gold) if gold in ranked else MAX_K

In [ ]:
def load_results(positions: list[Position]) -> pd.DataFrame:
    """The latest saved row per position; raises if a row names a position this build does not have."""
    parts = sorted(RESULTS_DIR.glob("part_*.parquet"))
    if not parts:
        return pd.DataFrame(columns=["pos_id", "error"])
    rows = pd.concat(map(pd.read_parquet, parts), ignore_index=True).drop_duplicates("pos_id", keep="last")
    expected = pd.DataFrame([(p.pos_id, p.user_id, p.token_index, p.gold) for p in positions],
                            columns=["pos_id", "user_id", "token_index", "gold"])
    joined = rows.merge(expected, on="pos_id", how="left", suffixes=("", "_expected"))
    mismatched = ~((joined.user_id == joined.user_id_expected) & (joined.token_index == joined.token_index_expected)
                   & (joined.gold == joined.gold_expected))
    if mismatched.any():
        raise ValueError(f"{mismatched.sum():,} saved rows in {RESULTS_DIR} do not match this build's positions")
    return rows


def run(positions: list[Position], max_calls: int | None, workers: int = MAX_WORKERS, every: int = 500) -> pd.DataFrame:
    """Score the first `max_calls` positions of one fixed shuffle; return one row per position of that subset.

    Positions already scored without an error are skipped.
    """
    order = list(positions)
    random.Random(0).shuffle(order)
    subset = order if max_calls is None else order[:max_calls]
    saved = load_results(positions)
    done = set(saved.pos_id[saved.error.isna()])
    todo = [p for p in subset if p.pos_id not in done]
    parts = sorted(RESULTS_DIR.glob("part_*.parquet"))
    part = int(parts[-1].stem.removeprefix("part_")) + 1 if parts else 0
    if todo:
        rows = []
        with ThreadPoolExecutor(workers) as pool:
            for i, row in enumerate(tqdm(pool.map(score_position, todo), total=len(todo)), start=1):
                rows.append(row)
                if len(rows) == every or i == len(todo):
                    pd.DataFrame(rows).to_parquet(RESULTS_DIR / f"part_{part:05d}.parquet")
                    rows, part = [], part + 1
    else:
        print(f"All {len(subset):,} positions are already scored.")
    results = load_results(positions)
    return results[results.pos_id.isin({p.pos_id for p in subset})]


results = run(positions, max_calls=MAX_CALLS)
succeeded = results[results.error.isna()]
print(f"Scored {len(results):,} positions with {len(results) - len(succeeded)} errors, which count as misses. "
      f"Median latency {succeeded.latency_s.median():.2f} s, p95 {succeeded.latency_s.quantile(0.95):.2f} s.")

## 4. Metrics

Everything below is a function of one integer per position: **`first_hit`, the 0-based rank of the
held-out item in the returned list, or 100 when it is not there.** Then, averaged over all positions:

- `recall@k` = share of positions with `first_hit < k` (with one answer per position this is also the
  hit rate).
- `ndcg@k` = `1 / log2(first_hit + 2)` when hit, else 0. The ideal DCG is 1, so no normalization is needed.
- `mrr@k` = `1 / (first_hit + 1)` when hit, else 0.

A miss counts as zero and is never dropped. Worked example: `first_hit = [0, 3, 100, 12]` gives
recall@10 = 2/4 = 0.5, ndcg@10 = (1 + 1/log2(5)) / 4 = 0.358, mrr@10 = (1 + 1/4) / 4 = 0.3125.

In [ ]:
def metrics(hits: np.ndarray, ks: tuple[int, ...] = KS) -> pd.Series:
    """Recall, NDCG and MRR at each k, averaged over positions; a miss counts as zero."""
    out = {"count": len(hits)}
    for k in ks:
        hit = hits < k
        out[f"recall@{k}"] = hit.mean()
        out[f"ndcg@{k}"] = np.where(hit, 1 / np.log2(hits + 2), 0).mean()
        out[f"mrr@{k}"] = np.where(hit, 1 / (hits + 1), 0).mean()
    return pd.Series(out)


example = metrics(np.array([0, 3, 100, 12]), ks=(1, 10, 100))
example[["recall@1", "recall@10", "recall@100", "ndcg@10", "mrr@10"]].to_frame("worked example").T.round(4)

### The floor: popularity

The catalog's `frequency` column counts how often each item was viewed in the training period. Ranking by
it gives every position the same top 100. Only recall is shown here; `metrics()` also returns NDCG and MRR
at every depth in `KS`.

In [ ]:
popular = catalog.select(["id", "frequency"]).to_pandas().sort_values("frequency", ascending=False)
top_popular = popular.id.head(MAX_K).tolist()
popularity_hits = np.array([first_hit(top_popular, p.gold) for p in positions])
metrics(popularity_hits)[["recall@1", "recall@10", "recall@100"]].to_frame("popularity").T.round(5)

## 5. Results

- `api`: this notebook's measurement.
- `api, 95 % interval`: a bootstrap over positions, which resamples the scored positions 500 times and
  recomputes the metrics. On a partial run it shows how far the subset can be trusted as an estimate of
  the full set.
- `paper`: the paper's number, with item ages and sales measured at each position's own date.
- `internal, same date`: our internal evaluation with one date for every position, 2022-01-29, the date
  the hosted catalog was embedded with.
- `popularity`: the floor from above.

In [ ]:
ROWS = ["recall@1", "recall@5", "recall@10", "recall@20", "recall@50", "recall@100", "ndcg@10", "mrr@10"]
hits = results.first_hit.to_numpy()  # a failed call counts as a miss
rng = np.random.default_rng(0)
boot = pd.DataFrame([metrics(rng.choice(hits, len(hits))) for _ in range(500)])

print(f"Results over {len(hits):,} of the {N_POSITIONS:,} positions.")
table = pd.DataFrame({
    "api": metrics(hits),
    "paper": pd.Series(PAPER, dtype=float),
    "internal, same date": pd.Series(SAME_DATE, dtype=float),
    "popularity": metrics(popularity_hits[results.pos_id.to_numpy()]),  # on the same positions as api
}).loc[ROWS]
low, high = boot.quantile(0.025), boot.quantile(0.975)
table.round(5).assign(**{"api, 95 % interval": [f"{low[r]:.5f} to {high[r]:.5f}" for r in ROWS]})[
    ["api", "api, 95 % interval", "paper", "internal, same date", "popularity"]]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.6))
for name, style in (("api", "-"), ("paper", "--"), ("internal, same date", "-."), ("popularity", ":")):
    ax.plot(PLOT_KS, table.loc[[f"recall@{k}" for k in PLOT_KS], name], style, marker="o", ms=5, lw=1.5, label=name)
ax.set(xscale="log", xlabel="k", ylabel="recall@k", xticks=PLOT_KS, xticklabels=PLOT_KS)
ax.grid(alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False);

Where does the model earn its recall? A position with no history is a cold start: the model can only
answer from the date, the profile and the catalog. A position at the start of a session sees earlier days
but nothing from the current one.

In [ ]:
context = np.where(results.n_hist == 0, "no history", np.where(results.at_session_start, "session start", "mid-session"))
results.assign(context=context).groupby("context").agg(
    **{"recall@10": ("first_hit", lambda h: (h < 10).mean()),
       "recall@100": ("first_hit", lambda h: (h < 100).mean()),
       "positions": ("first_hit", "size")}
).reindex(["no history", "session start", "mid-session"]).round(5)